Defining function

In [4]:
def tiff_to_parquet(file_name, input_folder, output_folder):
    import os
    import tifffile as tiff
    import numpy as np
    import geopandas as gpd
    from shapely.geometry import shape
    from rasterio.features import shapes
    import pandas as pd
    from collections import defaultdict
    from tqdm import tqdm
    from pathlib import Path

    tiff_path = os.path.join(input_folder, file_name)
    output_name = file_name.split('_DAPI')[0]
    output_dir = Path(output_folder) / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = output_dir / "nucleus_boundaries.parquet"

    print("Loading TIFF...")
    mask = tiff.imread(tiff_path)
    print('tiff loaded successfully')

    assert mask.ndim == 2, "Only 2D masks supported"
    assert mask.dtype.kind in ("i", "u"), "Mask must be integer-labeled"

    print("Extracting polygons...")

    # Collect all geometry parts per label
    geom_parts = defaultdict(list)
    num_shapes = len(np.unique(mask)) - 1

    for geom, value in tqdm(shapes(mask, mask=mask > 0), total=num_shapes, desc="Extracting shapes"):   #iterates through connected regions of same pixel values
        entity_id = int(value)
        poly = shape(geom) #shape(geom) doslovce izbaci polygon, us mislu onaj oblk stnaice 
        if poly.is_empty:
            continue

        # Normalize to polygons
        if poly.geom_type == "Polygon":
            geom_parts[entity_id].append(poly)
        elif poly.geom_type == "MultiPolygon":
            geom_parts[entity_id].extend(list(poly.geoms))
        
        #print(f"Processed EntityID: {entity_id}")

    print(f"Found {len(geom_parts)} unique nuclei")
    records = []

    '''
    #this is in case we want Multipolygon
    for entity_id, polygons in tqdm(geom_parts.items(), desc="Iterating through nuclei"):  #iterates through each unique label/cell
        multipolygon = MultiPolygon(polygons)
        centroid = multipolygon.centroid

        records.append({
            "EntityID": entity_id,
            "Geometry": multipolygon,
            "centroid_x": centroid.x,
            "centroid_y": centroid.y
            })
    '''

    for entity_id, polygons in tqdm(geom_parts.items(), desc="Iterating through nuclei"):
        for poly in polygons:
            centroid = poly.centroid

            records.append({
                "EntityID": entity_id,
                "geometry": poly,
                "centroid_x": centroid.x,
                "centroid_y": centroid.y
            })

    # Build GeoDataFrame
    gdf = gpd.GeoDataFrame(records, geometry="geometry",crs=None )

    # Enforce column order
    gdf = gdf[[
        "geometry",
        "EntityID",
        "centroid_x",
        "centroid_y"
        ]]

    # Removing multiple polygons per cell by keeping only the largest one
    gdf["area"] = gdf.geometry.area
    # Keep only the largest polygon per entity_id
    gdf_clean = gdf.loc[gdf.groupby("EntityID")["area"].idxmax()].copy()
    # drop helper column
    gdf_clean = gdf_clean.drop(columns="area")

    print(f"Writing Parquet... {parquet_path}")
    
    gdf_clean.to_parquet(parquet_path, index=False)
    print("Done.")
     

Main script for conerting tif to parquet, Kod nas je centrala

In [ ]:
import os
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
MC_SEGMENTATIONS_DIR = Path("")  # directory containing .tif segmentation masks
INPUT_DIR = Path("")              # output directory (input_for_segger)

# Get all .tif files
tif_files = [f for f in os.listdir(MC_SEGMENTATIONS_DIR) if f.endswith(".tif")]

# Loop through and convert each
for file_name in tif_files:
    tiff_to_parquet(file_name, str(MC_SEGMENTATIONS_DIR), str(INPUT_DIR))


In [ ]:
tiff_to_parquet('33156-Slide-1_A1-1_DAPI_cp_masks.tif', str(MC_SEGMENTATIONS_DIR), str(INPUT_DIR))


In [ ]:
#run tiff_to_parquet for all tiffs in the folder
tiff_to_parquet('', str(MC_SEGMENTATIONS_DIR), str(INPUT_DIR))


In [ ]:
import tifffile as tiff
mask = tiff.imread(MC_SEGMENTATIONS_DIR.parent / '33156-Slide-1_submission' / '33156-Slide-1_A1-1_DAPI.tiff')


In [5]:
import numpy as np
num_shapes = len(np.unique(mask)) - 1